In [4]:

# ==========================================================
# NLP EMOTION CLASSIFICATION
# TF-IDF / BoW + Naive Bayes / Logistic Regression
# ==========================================================

# 1. IMPORT LIBRARIES

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import string
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import (
    CountVectorizer,
    TfidfVectorizer,
    ENGLISH_STOP_WORDS
)
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report


# ==========================================================
# 2. LOAD DATASET
# ==========================================================

df = pd.read_csv(
    'train.txt',
    sep=';',
    header=None,
    names=['text', 'emotion']
)

print("Dataset shape:", df.shape)
print("\nFirst five records:")
print(df.head())

print("\nEmotion distribution:")
print(df['emotion'].value_counts())


# ==========================================================
# 3. HANDLE MISSING VALUES
# ==========================================================

df = df.dropna(subset=['text', 'emotion']).copy()

df['text'] = df['text'].astype(str)
df['emotion'] = df['emotion'].astype(str).str.strip()


# ==========================================================
# 4. ENCODE EMOTION LABELS
# ==========================================================

unique_emotion = sorted(df['emotion'].unique())

emotion_num = {
    emo: i for i, emo in enumerate(unique_emotion)
}

num_emotion = {
    i: emo for emo, i in emotion_num.items()
}

df['emotion'] = df['emotion'].map(emotion_num)

print("\nEmotion mapping:", emotion_num)
print("Reverse mapping:", num_emotion)

print("\nEncoded emotion distribution:")
print(df['emotion'].value_counts())

if df['emotion'].nunique() < 2:
    raise ValueError(
        "Dataset must contain at least two emotion classes."
    )

if df['emotion'].value_counts().min() < 2:
    raise ValueError(
        "Each emotion needs at least two records for this train/test split."
    )


# ==========================================================
# 5. TEXT PREPROCESSING
# ==========================================================

def remove_punc(text):
    return text.translate(
        str.maketrans('', '', string.punctuation)
    )


def remove_num(txt):
    new = ""

    for i in txt:
        if not i.isdigit():
            new += i

    return new


def remove_url(txt):
    return re.sub(r'https?://\S+|www\.\S+', '', txt)


def remove_html_tags(txt):
    return re.sub(r'<.*?>', '', txt)


def remove_emoji(txt):
    new = ""

    for i in txt:
        if i.isascii():
            new += i

    return new


stop_words = set(ENGLISH_STOP_WORDS)


def remove(txt):
    words = txt.split()

    new = []

    for w in words:
        if w not in stop_words:
            new.append(w)

    return ' '.join(new)


def clean_text(text):
    text = text.lower()
    text = remove_punc(text)
    text = remove_num(text)
    text = remove_url(text)
    text = remove_html_tags(text)
    text = remove_emoji(text)
    text = remove(text)

    return text


df['text'] = df['text'].apply(clean_text)

print("\nCleaned text:")
print(df[['text', 'emotion']].head())


# ==========================================================
# 6. TRAIN / TEST SPLIT
# ==========================================================

X = df['text']
y = df['emotion']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining records:", len(X_train))
print("Testing records:", len(X_test))


# ==========================================================
# 7. BAG OF WORDS + MULTINOMIAL NAIVE BAYES
# ==========================================================

bow_vectorizer = CountVectorizer()

X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

nb_model = MultinomialNB()

nb_model.fit(X_train_bow, y_train)

pred_bow = nb_model.predict(X_test_bow)

print("\nBoW + MultinomialNB accuracy:")
print(accuracy_score(y_test, pred_bow))

print("\nBoW Classification Report:")
print(
    classification_report(
        y_test,
        pred_bow,
        labels=sorted(num_emotion.keys()),
        target_names=[
            num_emotion[i] for i in sorted(num_emotion)
        ],
        zero_division=0
    )
)


# ==========================================================
# 8. TF-IDF + MULTINOMIAL NAIVE BAYES
# ==========================================================

tfidf_nb = TfidfVectorizer()

X_train_tfidf_nb = tfidf_nb.fit_transform(X_train)
X_test_tfidf_nb = tfidf_nb.transform(X_test)

nb2_model = MultinomialNB()

nb2_model.fit(X_train_tfidf_nb, y_train)

pred_tfidf_nb = nb2_model.predict(X_test_tfidf_nb)

print("\nTF-IDF + MultinomialNB accuracy:")
print(accuracy_score(y_test, pred_tfidf_nb))

print("\nTF-IDF + Naive Bayes Classification Report:")
print(
    classification_report(
        y_test,
        pred_tfidf_nb,
        labels=sorted(num_emotion.keys()),
        target_names=[
            num_emotion[i] for i in sorted(num_emotion)
        ],
        zero_division=0
    )
)


# ==========================================================
# 9. TF-IDF + LOGISTIC REGRESSION
# ==========================================================

tfidf = TfidfVectorizer()

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

lr = LogisticRegression(max_iter=1000)

lr.fit(X_train_tfidf, y_train)

log_pred = lr.predict(X_test_tfidf)

print("\nTF-IDF + Logistic Regression accuracy:")
print(accuracy_score(y_test, log_pred))

print("\nLogistic Regression Classification Report:")
print(
    classification_report(
        y_test,
        log_pred,
        labels=sorted(num_emotion.keys()),
        target_names=[
            num_emotion[i] for i in sorted(num_emotion)
        ],
        zero_division=0
    )
)


# ==========================================================
# 10. SAVE TRAINED MODEL, VECTORIZER AND LABEL MAP
# ==========================================================

with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

with open('emotion_model.pkl', 'wb') as f:
    pickle.dump(lr, f)

with open('label_map.pkl', 'wb') as f:
    pickle.dump(num_emotion, f)

print("\nTraining and saving successful!")

print("Saved files:")
print("1. tfidf_vectorizer.pkl")
print("2. emotion_model.pkl")
print("3. label_map.pkl")


# ==========================================================
# 11. LOAD SAVED FILES
# ==========================================================

with open('tfidf_vectorizer.pkl', 'rb') as f:
    tfidf_loaded = pickle.load(f)

with open('emotion_model.pkl', 'rb') as f:
    model_loaded = pickle.load(f)

with open('label_map.pkl', 'rb') as f:
    label_map_loaded = pickle.load(f)


# ==========================================================
# 12. TEST WITH SAMPLE TEXT
# ==========================================================

sample = "i am so happy today everything feels perfect"

sample_clean = clean_text(sample)

sample_vec = tfidf_loaded.transform([sample_clean])

pred = model_loaded.predict(sample_vec)[0]

print("\nSample text:", sample)
print("Cleaned text:", sample_clean)
print("Predicted emotion:", label_map_loaded[int(pred)])


# ==========================================================
# 13. FINAL DATASET CHECK
# ==========================================================

print("\nFinal dataset shape:", df.shape)

print("\nEmotion distribution:")
print(df['emotion'].value_counts())

print("\nUnique emotion values:")
print(df['emotion'].unique())

print("\nEmotion mapping:")
print(num_emotion)


Dataset shape: (10053, 2)

First five records:
                                                text  emotion
0                            i didnt feel humiliated  sadness
1  i can go from feeling so hopeless to so damned...  sadness
2   im grabbing a minute to post i feel greedy wrong    anger
3  i am ever feeling nostalgic about the fireplac...     love
4                               i am feeling grouchy    anger

Emotion distribution:
emotion
joy         3382
sadness     2906
anger       1356
fear        1199
love         830
surprise     379
Name: count, dtype: int64

Emotion mapping: {'anger': 0, 'fear': 1, 'joy': 2, 'love': 3, 'sadness': 4, 'surprise': 5}
Reverse mapping: {0: 'anger', 1: 'fear', 2: 'joy', 3: 'love', 4: 'sadness', 5: 'surprise'}

Encoded emotion distribution:
emotion
2    3382
4    2906
0    1356
1    1199
3     830
5     379
Name: count, dtype: int64

Cleaned text:
                                               text  emotion
0                             didnt fe